## Using User Defined Tool

### Installing Libraries

In [ ]:
%pip install anthropic python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key = anthropic_api_key)

### Defining the Tool Schema

In [ ]:
tools = [
    {
        "name": "GetCurrentWeather",
        "description": "Retrieve the current weather information for a specified location using the wttr.in weather service.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City or location to retrieve the weather for, for example Bangalore."
                }
            },
            "required": ["location"]
        }
    }
]

### Implementing the Weather API Call

In [ ]:
import requests

def get_current_weather(location: str):
    response = requests.get(
        f"https://wttr.in/{location}",
        params={
            "format": "j1"
        }
    )

    response.raise_for_status()

    return response.json()

### Calling the Messages API and Handling Agent Loop

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What's the weather in Bangalore today?"
    }
]

while True:
    response = client.messages.create(
        model = claude_model,
        max_tokens = 1024,
        tools = tools,
        tool_choice = {"type": "auto"},
        messages = messages
    )

    messages.append(
        {
            "role": "assistant",
            "content": response.content
        }
    )

    tool_used = False

    for block in response.content:
        if block.type == "text":
            print(block.text + "\n\n")

        elif block.type == "tool_use":
            tool_used = True

            if block.name == "GetCurrentWeather":
                print("Invoking GetCurrentWeather Tool \n\n")
                weather = get_current_weather(
                    block.input["location"]
                )

                messages.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": str(weather)
                            }
                        ]
                    }
                )

    if not tool_used:
        break